# 第4章 收益率计量 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch04_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch04_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 6：牛顿法 vs 二分法求 YTM


In [ ]:
import numpy as np
from fi.cashflow import make_cashflows
from fi.pricing import price_bond, ytm
from fi import plotting
plotting.use_chinese_style()
cfs, ts = make_cashflows(0.03, 3, 1, 100); price = 97.2249
def ytm_bisection(price, cfs, ts, freq=1, lo=-0.5, hi=1.0, tol=1e-12):
    it = 0
    while it < 500:
        it += 1; mid = 0.5*(lo+hi); p = price_bond(cfs, ts, mid, freq)
        if abs(p-price) < tol: return mid, it
        lo, hi = (mid, hi) if p > price else (lo, mid)
    return mid, it
y_newton = ytm(price, cfs, ts, freq=1, guess=0.30)   # 故意差初值
y_bis, n = ytm_bisection(price, cfs, ts)
print(f'牛顿法(差初值30%) YTM = {y_newton*100:.6f}%  几步即收敛')
print(f'二分法           YTM = {y_bis*100:.6f}%  迭代 {n} 次（稳健但慢）')


## 编程实验 7：实现复合收益率 vs 再投资利率


In [ ]:
rs = np.linspace(0, 0.06, 121)
rcy = [((sum(3*(1+r)**(3-t) for t in (1,2,3)) + 100)/100)**(1/3) - 1 for r in rs]
fig, ax = plotting.new_axes()
ax.plot(rs*100, np.array(rcy)*100)
ax.axhline(3, ls=':', color='gray'); ax.axvline(3, ls=':', color='gray')
ax.scatter([3],[3], color='k', zorder=5, label='再投资@YTM 时实现收益=YTM')
ax.set_xlabel('再投资利率 (%)'); ax.set_ylabel('实现复合收益率 (%)'); ax.set_title('再投资风险'); ax.legend()
fig.tight_layout()


## 编程实验 8：即期/远期曲线（内置；akshare 联网降级）


In [ ]:
from fi import data
from fi.pricing import forward_rate
curve = data.load_sample('cgb_yield_curve'); zt = dict(zip(curve['tenor'], curve['yield_pct']/100))
ten = list(curve['tenor']); fx, fy = [], []
for a,b in zip(ten[:-1], ten[1:]): fx.append(b); fy.append(forward_rate(lambda t: zt[t], a, b, 1)*100)
fig, ax = plotting.new_axes()
ax.plot(curve['tenor'], curve['yield_pct'], marker='o', label='即期(样本近似)')
ax.plot(fx, fy, marker='s', ls='--', label='隐含远期')
ax.set_xlabel('期限（年）'); ax.set_ylabel('利率 (%)'); ax.set_title('即期与远期曲线'); ax.legend()
fig.tight_layout()
print('akshare 真实数据：import akshare as ak; ak.bond_zh_us_rate() 取国债收益率后同样处理')
